# Reproducing the reference solution

For a bit of context, this notebook aims to reproduce the PINN solution in figure 1 of [the original paper](https://arxiv.org/pdf/2205.04611). 

To revisit the problem, figure 1 solves the 1D wave equation

$$\frac {1} {c^2} u_{tt} = u_{xx},$$

with $c = 1$, subject to Dirichlet initial and boundary conditions $g(t, x)$:

$$g(t, x) := g^n_i = \begin{cases}
\frac {\partial}{\partial t} u(0, x) &= \tfrac{1}{10}\pi k\left(\sin((x+0.5)\pi k) - \sin((x-0.5)\pi k)\right)\\[3pt]
u(0, x) &= \tfrac{1}{10}\left(\cos((x+0.5)\pi k) + \cos((x-0.5)\pi k)\right)\\[3pt]
u(t, 0) &= \tfrac{1}{10}\left(\cos((-t+0.5)\pi k) + \cos((t-0.5)\pi k)\right)\\[3pt]
u(t, L) &= \tfrac{1}{10}\left(\cos((L-t+0.5)\pi k) + \cos((L+t-0.5)\pi k)\right)\\[3pt]
\end{cases} \text{for } k = 1, 2, 3, 4, 5,$$

where $t \in (0, 1.0),$ $x \in (-1.0, 1.0)$.

## PINN solution

In the ODIL task, we discretised our governing equation and boundary condition and formulated a discrete PDE loss from the two. ODIL borrows that notion from PINNs, which can approximate PDE solutions using a neural network via loosely enforcing the PDE and BCs/ICs through the loss. The key difference is that PINNs sample a continuous loss function. Ultimately, our hope is that a neural network $\mathcal N(t, x, \mathbf \theta)$ with parameters $\bf \theta$, can approximate our PDE

$$\mathcal{N}(t, x, \mathbf{\theta}) \approx u(t, x).$$

As mentioned, when using neural networks to solve PDEs, it is the responsibility of the loss function to enforce the PDE and boundary/ initial conditions. The network outputs data at 'collocation points': points in the domain at which the neural network approximates the PDE, which is independent of the number of parameters. Collocation points are comprised of interior points, where you differentiate the network output and check the PDE residual, and boundary/ initial points, where we check the network output against the known solution.

### Formulating the loss

We form the loss in pretty much the same way as the PINN. We could also add in a data loss, but we won't do that for this case. We require:

- A PDE residual (which we hope tends to zero, enforcing the PDE)
- A boundary/ initial condition loss (which we compare against the reference)

Our PDE residual, evaluated at the $N_p$ interior points, is

$$\mathcal{R}_\text{PDE} = \underbrace{\frac{1}{N_p}\sum_i^{N_p}\left(\left[\frac{\partial^2}{\partial t^2} - \frac{\partial^2}{\partial x^2}\right]\mathcal{N}(t_i, x_i, \mathbf{\theta})\right)^2}_{\rightarrow 0 \text{ when satisfied}}.$$

Our boundary/ intitial conditions residual is

$$\mathcal{R}_\text{cond} = \left(\mathcal{N}(t_\Gamma, x_\Gamma, \mathbf{\theta}) - g(t_\Gamma, x_\Gamma)\right)^2,$$

for the boundary/ IC domain $\Gamma$.

Our full loss, written explicitly, is thus

$$\mathcal{L}(\boldsymbol{\theta}) = 
\left.
\begin{aligned}
  &\lambda_1\left(\mathcal{N}(t=0, x, \boldsymbol{\theta}) - \tfrac{1}{10}\left(\cos((x+0.5)\pi k) + \cos((x-0.5)\pi k)\right)\right)^2 \\
  &\lambda_2\left(\mathcal{N}(t, x=0, \boldsymbol{\theta}) - \tfrac{1}{10}\left(\cos((-t+0.5)\pi k) + \cos((t-0.5)\pi k)\right)\right)^2 \\
  &\lambda_3\left(\mathcal{N}(t, x=L, \boldsymbol{\theta}) - \tfrac{1}{10}\left(\cos((L-t+0.5)\pi k) + \cos((L+t-0.5)\pi k)\right)\right)^2 \\
  &\lambda_4\left(\frac{\partial}{\partial t}\mathcal{N}(t=0, x, \boldsymbol{\theta}) - \tfrac{1}{10}\pi k\left(\sin((x+0.5)\pi k) - \sin((x-0.5)\pi k)\right)\right)^2
\end{aligned}
\right\} \text{BC/ICs}
\\[6pt]
+ \left.\frac{\lambda_5}{N_p}\sum_i^{N_p}\left(\left[\frac{\partial^2}{\partial t^2} - \frac{\partial^2}{\partial x^2}\right]\mathcal{N}(t_i, x_i, \mathbf{\theta})\right)^2\right\} \text{PDE}
$$

where $k = 1, 2, 3, 4, 5$, and $\lambda_n$ are regularisation parameters.

## The network

The paper uses a simple feed forward network of 2 fully connected layers with 25 neurons each and a tanh activation. The PINN uses 8192 interior points and 768 boundary/ initial points as collocation points. Setting up the network should be straightforward, but the loss might take a bit of work. Note, to compute gradients of the neural network w.r.t. its inputs, we can use `torch.autograd.grad`. 